# **Import Library**

In [1]:
from pyspark.sql import *
from pyspark.sql.functions import *
import pandas as pd

# **Spark Session**

In [2]:
spark = SparkSession.builder.appName("Bank").getOrCreate()
df = spark.read.csv("bank_with_cx_name.csv",header = True, inferSchema = True)

df.show(5)
df.printSchema()

+--------------+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_00001| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
|Customer_00002| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
|Customer_00003| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
|Customer_00004| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -

# **Data Preprocessing**

In [61]:
from pyspark.sql.types import DoubleType, FloatType

df.select([
    count(when(col(c).isNull() | (isnan(col(c)) if isinstance(df.schema[c].dataType, (DoubleType, FloatType)) else False), c)).alias(c)
    for c in df.columns
]).show()

+-------+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+-------+
|cx_name|age|job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|cx_type|
+-------+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+-------+
|      0|  0|  0|      0|        0|      0|      0|      0|   0|      0|  0|    0|       0|       0|    0|       0|       0|      0|      0|
+-------+---+---+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+-------+



In [64]:
# duplicates

df = df.dropDuplicates(['cx_name'])

In [3]:
# Count total cx

df.count()


11162

# **Data Stats**

In [4]:
# Find cx with balance greater than 50,000


df.filter(col("balance")>50000).show()

+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|          job| marital|education|default|balance|housing|loan|  contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_02469| 61|self-employed|divorced| tertiary|     no|  52587|     no|  no| cellular| 10|  aug|     290|       1|   -1|       0| unknown|    yes|
|Customer_03044| 84|      retired| married|secondary|     no|  81204|     no|  no|telephone| 28|  dec|     679|       1|  313|       2|   other|    yes|
|Customer_03237| 61|self-employed|divorced| tertiary|     no|  52587|     no|  no| cellular| 15|  feb|     394|       3|  189|       1| success|    yes|
|Customer_03381| 84|      retired| married|secondary|     no|  81204|     no|  no|

In [6]:
# find cx who have both housing loan and personal loan

df.filter((col("housing")=="yes") & (col("loan")=="yes")).show()

+--------------+---+------------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|         job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+------------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_00006| 42|  management|  single| tertiary|     no|      0|    yes| yes|unknown|  5|  may|     562|       2|   -1|       0| unknown|    yes|
|Customer_00007| 56|  management| married| tertiary|     no|    830|    yes| yes|unknown|  6|  may|    1201|       1|   -1|       0| unknown|    yes|
|Customer_00013| 29|  management| married| tertiary|     no|    199|    yes| yes|unknown|  7|  may|    1689|       4|   -1|       0| unknown|    yes|
|Customer_00020| 49|      admin.|divorced|secondary|     no|    168|    yes| yes|unknown|  8|  may| 

In [10]:
# Find avg account balance by job

df.groupBy(col("job")).agg(round(avg(col("balance")),2)).show()

+-------------+----------------------+
|          job|round(avg(balance), 2)|
+-------------+----------------------+
|   management|               1793.66|
|      retired|               2417.25|
|      unknown|               1945.46|
|self-employed|               1865.37|
|      student|               1500.78|
|  blue-collar|               1203.93|
| entrepreneur|               1621.94|
|       admin.|               1195.87|
|   technician|               1556.29|
|     services|               1081.17|
|    housemaid|               1366.16|
|   unemployed|               1314.72|
+-------------+----------------------+



In [18]:
# Find deposit conversion rate by education

df.groupBy(col("education")).agg(round(count(when(col("deposit")=="yes",1))*100/count("*"),2)).show()

+---------+--------------------------------------------------------------------------+
|education|round(((count(CASE WHEN (deposit = yes) THEN 1 END) * 100) / count(1)), 2)|
+---------+--------------------------------------------------------------------------+
|  unknown|                                                                      50.7|
| tertiary|                                                                     54.11|
|secondary|                                                                     44.74|
|  primary|                                                                      39.4|
+---------+--------------------------------------------------------------------------+



In [19]:
# Top 10 cx with highest balance

df.orderBy(col("balance").desc()).limit(10).show()

+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|          job| marital|education|default|balance|housing|loan|  contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_03044| 84|      retired| married|secondary|     no|  81204|     no|  no|telephone| 28|  dec|     679|       1|  313|       2|   other|    yes|
|Customer_03381| 84|      retired| married|secondary|     no|  81204|     no|  no|telephone|  1|  apr|     390|       1|   94|       3| success|    yes|
|Customer_08209| 52|  blue-collar| married|  primary|     no|  66653|     no|  no| cellular| 14|  aug|     109|       3|   -1|       0| unknown|     no|
|Customer_10144| 43|       admin.|  single|secondary|     no|  56831|     no|  no|

In [21]:
# Find average call duration for successful deposits Solution

df.groupby(col("deposit")=='yes').agg(round(avg(col("duration")),2)).show()


+---------------+-----------------------+
|(deposit = yes)|round(avg(duration), 2)|
+---------------+-----------------------+
|           true|                 537.29|
|          false|                 223.13|
+---------------+-----------------------+



In [22]:
# Create Age Buckets

df.withColumn("age_bucket",
when(col("age")<25, "18-25")
.when(col("age")<35,"25-35")
.when(col('age')<45,"35-45")
.when(col('age')<55,"45-55")
.when(col('age')<65,"55-65")
.otherwise("65+")
).show()

+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|       cx_name|age|        job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|age_bucket|
+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|Customer_00001| 59|     admin.| married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|     55-65|
|Customer_00002| 56|     admin.| married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|     55-65|
|Customer_00003| 41| technician| married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|     35-45|
|Customer_00004| 55|   services| married

In [23]:
# Find customers never contacted before

df.filter(col("contact")=='unknown').show()

+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|        job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_00001| 59|     admin.| married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
|Customer_00002| 56|     admin.| married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
|Customer_00003| 41| technician| married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
|Customer_00004| 55|   services| married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579

In [24]:
df.show()

+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|       cx_name|age|        job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|Customer_00001| 59|     admin.| married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
|Customer_00002| 56|     admin.| married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
|Customer_00003| 41| technician| married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
|Customer_00004| 55|   services| married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579

In [25]:
# Count customers by marital status

df.groupby(col('marital')).count().show()

+--------+-----+
| marital|count|
+--------+-----+
|divorced| 1293|
| married| 6351|
|  single| 3518|
+--------+-----+



In [27]:
# Average balance by deposit status

df.groupby(col('deposit')).agg(round(avg(col('balance').alias("avg_balance")),2)).show()

+-------+-------------------------------------+
|deposit|round(avg(balance AS avg_balance), 2)|
+-------+-------------------------------------+
|     no|                              1280.23|
|    yes|                              1804.27|
+-------+-------------------------------------+



In [30]:
# Find top 5 jobs contributing maximum deposits

df.filter(col("deposit")=='yes')\
.groupBy(col('job'))\
.count()\
.orderBy(col('count').desc())\
.show(5)

+-----------+-----+
|        job|count|
+-----------+-----+
| management| 1301|
| technician|  840|
|blue-collar|  708|
|     admin.|  631|
|    retired|  516|
+-----------+-----+
only showing top 5 rows


In [33]:
# Find campaign effectiveness

df.groupby(col("campaign"))\
.agg(count('*').alias("cx"),
count(when(col('deposit')=='yes',1)).alias("converted"))\
.orderBy('campaign')\
.show()

+--------+----+---------+
|campaign|  cx|converted|
+--------+----+---------+
|       1|4798|     2561|
|       2|3028|     1401|
|       3|1321|      618|
|       4| 771|      317|
|       5| 378|      139|
|       6| 265|       92|
|       7| 139|       47|
|       8| 128|       32|
|       9|  72|       21|
|      10|  52|       14|
|      11|  40|       16|
|      12|  29|        4|
|      13|  30|        6|
|      14|  15|        4|
|      15|  13|        4|
|      16|   9|        2|
|      17|  14|        6|
|      18|   8|        0|
|      19|   5|        0|
|      20|   5|        1|
+--------+----+---------+
only showing top 20 rows


In [35]:
# Create a new column for High Value Customer


df = df.withColumn("cx_type",
when(col("balance")>=5000,"High Value")
.otherwise("Normal"))

df.show()

+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|       cx_name|age|        job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|   cx_type|
+--------------+---+-----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|Customer_00001| 59|     admin.| married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|    Normal|
|Customer_00002| 56|     admin.| married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|    Normal|
|Customer_00003| 41| technician| married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|    Normal|
|Customer_00004| 55|   services| married

In [37]:
# Rank customers by balance within each job

WindowSpec = Window.partitionBy(col('job')).orderBy(col('balance').desc())

df.withColumn(
  "rank",
  row_number().over(WindowSpec)
).show()

+--------------+---+------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+----------+----+
|       cx_name|age|   job| marital|education|default|balance|housing|loan|  contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|   cx_type|rank|
+--------------+---+------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+----------+----+
|Customer_10144| 43|admin.|  single|secondary|     no|  56831|     no|  no|  unknown| 15|  may|     243|       1|   -1|       0| unknown|     no|High Value|   1|
|Customer_02259| 39|admin.|  single| tertiary|     no|  22086|     no|  no| cellular|  4|  jun|     160|       1|   -1|       0| unknown|    yes|High Value|   2|
|Customer_06864| 32|admin.| married|secondary|     no|  20011|     no|  no| cellular| 20|  apr|     234|       1|   -1|       0| unknown|     no|High Value|   3|
|Customer_04192| 27|admin.| 

In [39]:
# Find duplicate customer names

df.groupBy(col("cx_name"))\
.count()\
.filter(col("count")>1)\
.show()

+-------+-----+
|cx_name|count|
+-------+-----+
+-------+-----+



In [40]:
# Find monthly deposit trend

df.groupby('month')\
.agg(
count(when(col('deposit')=='yes',1)).alias("deposits"))\
.orderBy('month')\
.show()

+-----+--------+
|month|deposits|
+-----+--------+
|  apr|     577|
|  aug|     688|
|  dec|     100|
|  feb|     441|
|  jan|     142|
|  jul|     627|
|  jun|     546|
|  mar|     248|
|  may|     925|
|  nov|     403|
|  oct|     323|
|  sep|     269|
+-----+--------+



In [42]:
# Find customers who were contacted previously but still didn't subscribe

df.filter(
    (col('previous')>0) & (col('deposit') == 'no')).show()

+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|       cx_name|age|          job| marital|education|default|balance|housing|loan|  contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|   cx_type|
+--------------+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+----------+
|Customer_05292| 48|  blue-collar| married|secondary|     no|    238|    yes| yes| cellular|  2|  jun|     118|       2|   81|       1| success|     no|    Normal|
|Customer_05293| 34|       admin.|  single|secondary|     no|    673|    yes|  no| cellular| 29|  jan|      89|       1|  260|       2| failure|     no|    Normal|
|Customer_05297| 31|  blue-collar| married|  primary|     no|   -199|    yes| yes|telephone| 12|  may|     149|       5|  299|      10| failure|     no|    Normal|
|Customer_05328|

In [44]:
# Build a Customer Summary Table (Real ETL Task)

summary = df.groupby(col('job'))\
.agg(
count("*").alias('total_cx'),
avg('balance').alias('avg_balance'),
max('balance').alias('max_balance'),
min('balance').alias('min_balance'),
count(when(col('deposit')=='yes',1)).alias('deposit_cx'),
count(when(col('deposit')=='no',1)).alias('no_deposit_cx')
)

summary.show()

+-------------+--------+------------------+-----------+-----------+----------+-------------+
|          job|total_cx|       avg_balance|max_balance|min_balance|deposit_cx|no_deposit_cx|
+-------------+--------+------------------+-----------+-----------+----------+-------------+
|   management|    2566|1793.6636788776304|      36252|      -6847|      1301|         1265|
|      retired|     778|2417.2506426735217|      81204|      -1206|       516|          262|
|      unknown|      70|1945.4571428571428|      19268|          0|        34|           36|
|self-employed|     405|1865.3728395061728|      52587|      -3058|       187|          218|
|      student|     360|1500.7833333333333|      23878|          0|       269|           91|
|  blue-collar|    1944|1203.9264403292182|      66653|      -1489|       708|         1236|
| entrepreneur|     328|1621.9420731707316|      51439|      -1965|       123|          205|
|       admin.|    1334|1195.8665667166417|      56831|      -1415|   

# **Analytics**

In [50]:
# Which customer segment has the highest deposit conversion rate?

df.groupby(col('job'))\
.agg(count("*").alias('total_cx'),
     count(when(col('deposit')=='yes',1)).alias('converted_cx'))\
     .withColumn('converion_rate',round(col('converted_cx')*100/col('total_cx'),2))\
    .orderBy(col('converion_rate').desc()).show()

+-------------+--------+------------+--------------+
|          job|total_cx|converted_cx|converion_rate|
+-------------+--------+------------+--------------+
|      student|     360|         269|         74.72|
|      retired|     778|         516|         66.32|
|   unemployed|     357|         202|         56.58|
|   management|    2566|        1301|          50.7|
|      unknown|      70|          34|         48.57|
|       admin.|    1334|         631|          47.3|
|self-employed|     405|         187|         46.17|
|   technician|    1823|         840|         46.08|
|     services|     923|         369|         39.98|
|    housemaid|     274|         109|         39.78|
| entrepreneur|     328|         123|          37.5|
|  blue-collar|    1944|         708|         36.42|
+-------------+--------+------------+--------------+



In [52]:
df2 = df.withColumn(
    'balance_bucket',
    when(col('balance')<1000,'Low')
    .when(col('balance')<3000,'Medium')
    .when(col('balance')<5000,'High')
    .otherwise('Very High')
)

result = df2.groupby('balance_bucket')\
.agg(
    count("*").alias('customers'),
    count(when(col('deposit')=='yes',1)).alias('coverted')
)\
.withColumn(
    "conversion_rate",
    round(col('coverted')*100/col('customers'),2)
)

result.show()

+--------------+---------+--------+---------------+
|balance_bucket|customers|coverted|conversion_rate|
+--------------+---------+--------+---------------+
|          High|      807|     476|          58.98|
|     Very High|      771|     441|           57.2|
|           Low|     7115|    3036|          42.67|
|        Medium|     2469|    1336|          54.11|
+--------------+---------+--------+---------------+



In [54]:
# Which marketing channel performs best?

result = df.groupby(col('contact'))\
.agg(
    count("*").alias('total_contacted'),
    count(when(col('deposit')=='yes',1)).alias('converted_cx')
)\
.withColumn(
    'success_rate',
    round(col('converted_cx')*100/col('total_contacted'),2)
)\
.orderBy(col('success_rate').desc())

result.show()

+---------+---------------+------------+------------+
|  contact|total_contacted|converted_cx|success_rate|
+---------+---------------+------------+------------+
| cellular|           8042|        4369|       54.33|
|telephone|            774|         390|       50.39|
|  unknown|           2346|         530|       22.59|
+---------+---------------+------------+------------+



In [55]:
# What is the ideal number of campaign calls?

result = df.groupby('campaign')\
.agg(
    count('*').alias('total_cx'),
    count(when(col('deposit')=='yes',1)).alias('converted_cx')
)\
.withColumn(
    'conversion_rate',
    round(col('converted_cx')*100/col('total_cx'),2)
)\
.orderBy('campaign')

result.show()


+--------+--------+------------+---------------+
|campaign|total_cx|converted_cx|conversion_rate|
+--------+--------+------------+---------------+
|       1|    4798|        2561|          53.38|
|       2|    3028|        1401|          46.27|
|       3|    1321|         618|          46.78|
|       4|     771|         317|          41.12|
|       5|     378|         139|          36.77|
|       6|     265|          92|          34.72|
|       7|     139|          47|          33.81|
|       8|     128|          32|           25.0|
|       9|      72|          21|          29.17|
|      10|      52|          14|          26.92|
|      11|      40|          16|           40.0|
|      12|      29|           4|          13.79|
|      13|      30|           6|           20.0|
|      14|      15|           4|          26.67|
|      15|      13|           4|          30.77|
|      16|       9|           2|          22.22|
|      17|      14|           6|          42.86|
|      18|       8| 

In [58]:
# Which age group generates the highest deposit conversion?

df2 = df.withColumn(
    'age',
    when(col('age')<25,'18-25')
    .when(col('age')<35,'25-35')
    .when(col('age')<45,'35-45')
    .when(col('age')<55,'45-55')
    .when(col('age')<65,'55-65')
    .otherwise('65+')\
)
result = df2.groupby(col('age'))\
.agg(
    count('*').alias('total_cx'),
    count(when(col('deposit')=='yes',1)).alias('converted_cx')
)\
.withColumn(
    'conversion_rate',
    round(col('converted_cx')*100/col('total_cx'),2)
)\
.orderBy('conversion_rate').show()

+-----+--------+------------+---------------+
|  age|total_cx|converted_cx|conversion_rate|
+-----+--------+------------+---------------+
|35-45|    3366|        1404|          41.71|
|45-55|    2205|         923|          41.86|
|25-35|    3628|        1773|          48.87|
|55-65|    1256|         641|          51.04|
|18-25|     282|         207|           73.4|
|  65+|     425|         341|          80.24|
+-----+--------+------------+---------------+



## **Trend Analysis**

In [59]:
# Monthly Deposit Conversion Trend

from pyspark.sql.functions import col,count,when,round

result=df.groupBy("month")\
.agg(
count("*").alias("total_customers"),
count(when(col("deposit")=="yes",1)).alias("total_deposits")
)\
.withColumn(
"conversion_pct",
round(col("total_deposits")*100/col("total_customers"),2)
)

month_order=["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]

result=result.withColumn(
"month_order",
when(col("month")=="jan",1)
.when(col("month")=="feb",2)
.when(col("month")=="mar",3)
.when(col("month")=="apr",4)
.when(col("month")=="may",5)
.when(col("month")=="jun",6)
.when(col("month")=="jul",7)
.when(col("month")=="aug",8)
.when(col("month")=="sep",9)
.when(col("month")=="oct",10)
.when(col("month")=="nov",11)
.otherwise(12)
).orderBy("month_order")

result.show()

+-----+---------------+--------------+--------------+-----------+
|month|total_customers|total_deposits|conversion_pct|month_order|
+-----+---------------+--------------+--------------+-----------+
|  jan|            344|           142|         41.28|          1|
|  feb|            776|           441|         56.83|          2|
|  mar|            276|           248|         89.86|          3|
|  apr|            923|           577|         62.51|          4|
|  may|           2824|           925|         32.75|          5|
|  jun|           1222|           546|         44.68|          6|
|  jul|           1514|           627|         41.41|          7|
|  aug|           1519|           688|         45.29|          8|
|  sep|            319|           269|         84.33|          9|
|  oct|            392|           323|          82.4|         10|
|  nov|            943|           403|         42.74|         11|
|  dec|            110|           100|         90.91|         12|
+-----+---